In [ ]:
import torch
import json
import re
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

In [ ]:
device="cuda"
model_name = "merged_model"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
MAX_LENGTH = 1024
SYSTEM_PROMPT = """You are an expert in thermodynamics and vapor-liquid equilibrium.
Your task is to predict equilibrium properties of binary mixtures from molecular structures and operating conditions.
Return only the requested numerical results.
"""
def build_messages(user_input, assistant_output=None):
    # 将System Prompt直接拼接到User内容
    user_content = SYSTEM_PROMPT.strip() + "\n\n" + user_input.strip()
    messages = [
        {
            "role": "user",
            "content": user_content
        }
    ]
    if assistant_output is not None:
        messages.append(
            {
                "role": "assistant",
                "content": assistant_output
            }
        )
    return messages

In [ ]:
def get_task_type(input_text):
    """
    根据input自动判断任务类型
    """
    text = input_text.lower()
    if "isobaric" in text:
        return "isobaric"
    elif "isothermal" in text:
        return "isothermal"
    else:
        raise ValueError("Cannot determine task type.")
def predict(example, model, tokenizer):
    model.eval()
    messages = build_messages(
        user_input=example["input"],
        assistant_output=None
    )
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True
    ).to(device)
    input_length = inputs["input_ids"].shape[1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    ).strip()
    return response
def parse_vle_output(text, task_type):
    text = str(text)
    y1 = None
    T = None
    P = None
    m = re.search(
        r"Vapor-phase mole fraction of component 1\s*:\s*([-+]?\d*\.?\d+)",
        text,
        re.I
    )
    if m:
        y1 = float(m.group(1))
    if task_type == "isobaric":
        m = re.search(
            r"Equilibrium temperature\s*:\s*([-+]?\d*\.?\d+)",
            text,
            re.I
        )
        if m:
            T = float(m.group(1))
    else:
        m = re.search(
            r"Equilibrium pressure\s*:\s*([-+]?\d*\.?\d+)",
            text,
            re.I
        )
        if m:
            P = float(m.group(1))
    return y1, T, P
def batch_predict_mixed_vle(
    test_file_path,
    save_path,
    max_samples=None
):
    results = []
    # Overall
    true_y1_all = []
    pred_y1_all = []
    # Isobaric
    true_y1_isoB = []
    pred_y1_isoB = []
    true_T_isoB = []
    pred_T_isoB = []
    # Isothermal
    true_y1_isoT = []
    pred_y1_isoT = []
    true_P_isoT = []
    pred_P_isoT = []
    with open(test_file_path, "r", encoding="utf-8") as f:
        lines = [line for line in f if line.strip()]
    if max_samples is not None:
        lines = lines[:max_samples]
    print(f"🚀 Total samples: {len(lines)}")
    model.eval()
    for i, line in enumerate(tqdm(lines)):
        try:
            data = json.loads(line)
            # 自动判断任务
            task_type = get_task_type(data["input"])
            # 模型预测
            raw_pred = predict(
                data,
                model,
                tokenizer
            )
            # 解析预测
            pred_y1, pred_T, pred_P = parse_vle_output(
                raw_pred,
                task_type
            )
            # 解析真实值
            true_y1, true_T, true_P = parse_vle_output(
                data["output"],
                task_type
            )
            #############################
            # 误差
            #############################
            err_y1 = None
            err_T = None
            err_P = None
            if pred_y1 is not None and true_y1 is not None:
                err_y1 = abs(pred_y1 - true_y1)
                true_y1_all.append(true_y1)
                pred_y1_all.append(pred_y1)
                if task_type == "isobaric":
                    true_y1_isoB.append(true_y1)
                    pred_y1_isoB.append(pred_y1)
                else:
                    true_y1_isoT.append(true_y1)
                    pred_y1_isoT.append(pred_y1)
            if task_type == "isobaric":
                if pred_T is not None and true_T is not None:
                    err_T = abs(pred_T - true_T)
                    true_T_isoB.append(true_T)
                    pred_T_isoB.append(pred_T)
            else:
                if pred_P is not None and true_P is not None:
                    err_P = abs(pred_P - true_P)
                    true_P_isoT.append(true_P)
                    pred_P_isoT.append(pred_P)

            #############################
            # 保存结果
            #############################
            result = {
                "task_type": task_type,
                "input": data["input"],
                "true_output": data["output"],
                "raw_prediction": raw_pred,
                "true_Y1": true_y1,
                "pred_Y1": pred_y1,
                "err_Y1": err_y1,
                "true_T": true_T,
                "pred_T": pred_T,
                "err_T": err_T,
                "true_P": true_P,
                "pred_P": pred_P,
                "err_P": err_P

            }
            results.append(result)
            #########################################
            print("\n" + "=" * 100)
            print(f"[{i+1}/{len(lines)}]")
            print("Task :", task_type)
            print("\nInput:")
            print(data["input"])
            print("\nGround Truth:")
            print(data["output"])
            print("\nPrediction:")
            print(raw_pred)
            print("\nParsed:")
            print(
                f"Y1 : {true_y1} -> {pred_y1}"
            )
            if task_type == "isobaric":
                print(
                    f"T  : {true_T} -> {pred_T}"
                )
            else:
                print(
                    f"P  : {true_P} -> {pred_P}"
                )
        except Exception as e:
            print(f"Sample {i+1} failed: {e}")
    #########################################
    # 保存json
    #########################################
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(
            results,
            f,
            indent=2,
            ensure_ascii=False
        )
    print(f"\nResults saved to {save_path}")
    #########################################
    # Metrics
    #########################################
    def print_metrics(name, y_true, y_pred):
        if len(y_true) == 0:
            print(f"\n{name}: No valid samples")
            return
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        mae = np.mean(np.abs(y_true - y_pred))
        rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
        r2 = r2_score(y_true, y_pred)
        print("\n" + "=" * 60)
        print(name)
        print(f"Samples : {len(y_true)}")
        print(f"MAE  : {mae:.6f}")
        print(f"RMSE : {rmse:.6f}")
        print(f"R²   : {r2:.6f}")
    print_metrics(
        "Overall Y1",
        true_y1_all,
        pred_y1_all
    )
    print_metrics(
        "Isobaric Y1",
        true_y1_isoB,
        pred_y1_isoB
    )
    print_metrics(
        "Isobaric Temperature",
        true_T_isoB,
        pred_T_isoB
    )
    print_metrics(
        "Isothermal Y1",
        true_y1_isoT,
        pred_y1_isoT
    )
    print_metrics(
        "Isothermal Pressure",
        true_P_isoT,
        pred_P_isoT
    )
    return results
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

In [ ]:
test_file_path = "../../按体系划分/数据/test_data_82_fine.jsonl"
save_path="./test_data_82_fine.json"
results_all = batch_predict_mixed_vle(
    test_file_path=test_file_path,
    save_path=save_path,
    max_samples= None
)

In [ ]:
import pandas as pd
import re
import numpy as np
# =========================
# 1. 转成DataFrame
# =========================
df = pd.DataFrame(results_all)
# =========================
# 2. 从input提取信息
# =========================
def extract_info(input_text):
    input_text = str(input_text)
    smiles1 = None
    smiles2 = None
    x1 = None
    pressure = None
    temperature = None
    # Component 1
    m = re.search(
        r"Component 1 SMILES:\s*(.+)",
        input_text,
        re.IGNORECASE
    )
    if m:
        smiles1 = m.group(1).split("\n")[0].strip()
    # Component 2
    m = re.search(
        r"Component 2 SMILES:\s*(.+)",
        input_text,
        re.IGNORECASE
    )
    if m:
        smiles2 = m.group(1).split("\n")[0].strip()
    # Liquid composition
    m = re.search(
        r"Liquid-phase mole fraction of component 1:\s*([-+]?\d*\.?\d+)",
        input_text,
        re.IGNORECASE
    )
    if m:
        x1 = float(m.group(1))
    # Pressure
    m = re.search(
        r"Operating pressure:\s*([-+]?\d*\.?\d+)",
        input_text,
        re.IGNORECASE
    )
    if m:
        pressure = float(m.group(1))
    # Temperature
    m = re.search(
        r"Operating temperature:\s*([-+]?\d*\.?\d+)",
        input_text,
        re.IGNORECASE
    )
    if m:
        temperature = float(m.group(1))
    return pd.Series(
        [
            smiles1,
            smiles2,
            x1,
            pressure,
            temperature
        ],
        index=[
            "SMILES_1",
            "SMILES_2",
            "X1",
            "Input_P(kPa)",
            "Input_T(K)"
        ]
    )
# 提取信息
df[
    [
        "SMILES_1",
        "SMILES_2",
        "X1",
        "Input_P(kPa)",
        "Input_T(K)"
    ]
] = df["input"].apply(extract_info)
# =========================
# 3. 调整列顺序
# =========================
df_save = pd.DataFrame()
df_save["Task"] = df["task_type"]
df_save["SMILES_1"] = df["SMILES_1"]
df_save["SMILES_2"] = df["SMILES_2"]
df_save["Input_X1"] = df["X1"]
#df_save["Input_P(kPa)"] = df["Input_P(kPa)"]
#df_save["Input_T(K)"] = df["Input_T(K)"]
df_save["True_Y1"] = df["true_Y1"]
df_save["Pred_Y1"] = df["pred_Y1"]
df_save["Error_Y1"] = df["err_Y1"]
df_save["True_T(K)"] = df["true_T"]
df_save["Pred_T(K)"] = df["pred_T"]
df_save["Error_T(K)"] = df["err_T"]
#df_save["True_P(kPa)"] = df["true_P"]
#df_save["Pred_P(kPa)"] = df["pred_P"]
#df_save["Error_P(kPa)"] = df["err_P"]
df_save["Raw Prediction"] = df["raw_prediction"]
# ========================
# 4. 保存Excel
# =========================
save_path = "./按体系划分/结果文件/调用的测试结果/test_data_82_fine.xlsx"
df_save.to_excel(
    save_path,
    index=False
)
print(f"保存成功：{save_path}")
df_save